In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import jiwer
from sklearn.model_selection import train_test_split
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, PreTrainedTokenizerFast

print("=" * 70)
print("Zero-Shot Evaluation: True Generalization Capability Validation")
print("=" * 70)

# ===========================
# 1. Paths and Configurations
# ===========================
PARQUET_PATH = "/root/autodl-fs/Manchu_OCR/train.parquet"
IMAGE_DIR = "/root/autodl-fs/Manchu_OCR/train"
MODEL_PATH = "/root/trocr_manchu_phase2_full"  # Phase 2 model weights path
MAX_SAMPLES = 500  # Randomly sample 500 unseen images for a quicker evaluation

# ===========================
# 2. Reproduce Split and Isolate Unseen Vocabulary
# ===========================
print("\n[1/4] Loading data and isolating seen vocabulary...")
df = pd.read_parquet(PARQUET_PATH)

# Reproduce the random split used during Phase 2 training
train_df, eval_df = train_test_split(df, test_size=0.1, random_state=42)

# Identify all vocabulary seen by the model during training
seen_words = set(train_df['roman'].unique())
print(f"  > Vocabulary size seen in training set: {len(seen_words)}")

# Core step: Retain only completely unseen words in the validation set
unseen_eval_df = eval_df[~eval_df['roman'].isin(seen_words)].reset_index(drop=True)
print(f"  > Number of strictly unseen images in validation set: {len(unseen_eval_df)}")

# Sample for evaluation to save time
if len(unseen_eval_df) > MAX_SAMPLES:
    test_df = unseen_eval_df.sample(n=MAX_SAMPLES, random_state=42).reset_index(drop=True)
else:
    test_df = unseen_eval_df
print(f"  > Evaluating {len(test_df)} completely unseen images for Zero-Shot performance.")

# ===========================
# 3. Load Model and Processor (Robust Version)
# ===========================
print(f"\n[2/4] Loading model from: {MODEL_PATH} ...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    # 1. Load Microsoft's base image processor
    base_processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
    
    # 2. Load the local Manchu tokenizer
    tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_PATH)
    
    # 3. Assemble into a custom Manchu Processor
    processor = TrOCRProcessor(image_processor=base_processor.image_processor, tokenizer=tokenizer)
    
    # 4. Load model weights
    model = VisionEncoderDecoderModel.from_pretrained(MODEL_PATH).to(device)
    model.eval()
    print("  > Model and Processor assembled and loaded successfully.")

except Exception as e:
    print(f"\nError: Failed to load model. Details: {e}")
    raise RuntimeError("Model loading failed. Please check the paths before proceeding.")


# ===========================
# 4. Inference and Evaluation
# ===========================
print("\n[3/4] Starting zero-shot inference...\n")

predictions = []
references = []
results_log = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Evaluating"):
    image_path = os.path.join(IMAGE_DIR, row['filename'])
    true_text = str(row['roman']).strip()
    
    try:
        image = Image.open(image_path).convert("RGB")
    except:
        continue  # Skip if the image is corrupted
        
    pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values, 
            max_length=64,
            num_beams=4,  # Enable beam search to improve decoding quality
            early_stopping=True
        )
        
    pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    
    predictions.append(pred_text)
    references.append(true_text)
    
    # Log the first 20 results for demonstration
    if len(results_log) < 20:
        local_cer = jiwer.cer(true_text, pred_text)
        results_log.append({
            "Truth": true_text,
            "Pred": pred_text,
            "CER": round(local_cer, 3),
            "Match": "Yes" if true_text == pred_text else "No"
        })

# ===========================
# 5. Calculate Final Scores and Display
# ===========================
print("\n[4/4] Evaluation complete. Calculating final scores...\n")

final_cer = jiwer.cer(references, predictions)
exact_matches = sum([1 for p, t in zip(predictions, references) if p == t])
word_acc = exact_matches / len(references)

print("=" * 60)
print("[Zero-Shot Evaluation Final Results]")
print("=" * 60)
print(f"Overall Character Error Rate (CER): {final_cer * 100:.2f}%")
print(f"Overall Word Accuracy (ACC): {word_acc * 100:.2f}%")
print("-" * 60)

print("[Top 20 Samples Close Observation]")
print(f"{'Truth':<20} | {'Prediction':<20} | {'CER':<6} | {'Match'}")
print("-" * 60)
for res in results_log:
    print(f"{res['Truth']:<20} | {res['Pred']:<20} | {res['CER']:<6} | {res['Match']}")
print("=" * 60)



Zero-Shot Evaluation: True Generalization Capability Validation

[1/4] Loading data and isolating seen vocabulary...
  > Vocabulary size seen in training set: 44355
  > Number of strictly unseen images in validation set: 3948
  > Evaluating 500 completely unseen images for Zero-Shot performance.

[2/4] Loading model from: /root/trocr_manchu_phase2_full ...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


  > Model and Processor assembled and loaded successfully.

[3/4] Starting zero-shot inference...



Evaluating:   0%|          | 0/500 [00:00<?, ?it/s]


[4/4] Evaluation complete. Calculating final scores...

[Zero-Shot Evaluation Final Results]
Overall Character Error Rate (CER): 7.68%
Overall Word Accuracy (ACC): 71.80%
------------------------------------------------------------
[Top 20 Samples Close Observation]
Truth                | Prediction           | CER    | Match
------------------------------------------------------------
acalara              | acalara              | 0.0    | Yes
feherehebihe         | feherehebihe         | 0.0    | Yes
toktobucina          | toktobucina          | 0.0    | Yes
duhenggele           | duhenggele           | 0.0    | Yes
isihidabucina        | isihibucina          | 0.154  | No
solinjihakv          | solinjihakv          | 0.0    | Yes
asharangge           | asharangge           | 0.0    | Yes
singgirafi           | singgirafi           | 0.0    | Yes
jerguweletele        | jerguweletele        | 0.0    | Yes
gerisehakvngge       | gerisehakvngge       | 0.0    | Yes
multufi              

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import jiwer
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, PreTrainedTokenizerFast


TEST_PARQUET_PATH = "/root/autodl-fs/Manchu_OCR/test.parquet"
IMAGE_DIR = "/root/autodl-fs/Manchu_OCR/test"  # Ensure this is the independent test folder
MODEL_PATH = "Um1neko/TrOCR_Manchu"  # Phase 2 model weights path

print(f"\n[1/4] Loading test set data: {TEST_PARQUET_PATH} ...")
test_df = pd.read_parquet(TEST_PARQUET_PATH)

print(f"  > Test set loaded successfully. Total images: {len(test_df)}")




print(f"\n[2/4] Loading model from: {MODEL_PATH} ...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    # 1. Load image processor
    base_processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
    
    # 2. Load the local Manchu tokenizer
    tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_PATH)
    
    # 3. Assemble into a custom Manchu Processor
    processor = TrOCRProcessor(image_processor=base_processor.image_processor, tokenizer=tokenizer)
    
    # 4. Load model weights
    model = VisionEncoderDecoderModel.from_pretrained(MODEL_PATH).to(device)
    model.eval()
    print("  > Model and Processor assembled and loaded successfully.")

except Exception as e:
    print(f"\nError: Failed to load model. Details: {e}")
    raise RuntimeError("Model loading failed. Please check the model path.")


print("\n[3/4] Starting inference...\n")

predictions = []
references = []
results_log = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Evaluating"):
    image_path = os.path.join(IMAGE_DIR, row['filename'])
    true_text = str(row['roman']).strip()
    
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        print(f"\nWarning: Cannot read image {image_path}, skipping. Error: {e}")
        continue
        
    pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values, 
            max_length=64,
            num_beams=4, # Enable beam search to improve decoding quality (TrOCR relies heavily on beam search)
            early_stopping=True
        )
        
    pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    
    predictions.append(pred_text)
    references.append(true_text)
    
    # Log the first 20 results for manual inspection
    if len(results_log) < 20:
        local_cer = jiwer.cer(true_text, pred_text)
        results_log.append({
            "Truth": true_text,
            "Pred": pred_text,
            "CER": round(local_cer, 3),
            "Match": "Yes" if true_text == pred_text else "No"
        })


print("\n[4/4] Evaluation complete. Calculating final scores...\n")

if len(predictions) == 0:
    print("Error: No samples were successfully processed. Please check the image paths.")
else:
    final_cer = jiwer.cer(references, predictions)
    final_wer = jiwer.wer(references, predictions)
    exact_matches = sum([1 for p, t in zip(predictions, references) if p == t])
    word_acc = exact_matches / len(references)

    print("=" * 60)
    print("[TrOCR Test Set Final Results]")
    print("=" * 60)
    print(f"Overall Character Error Rate (CER): {final_cer * 100:.4f}%")
    print(f"Overall Word Error Rate (WER)     : {final_wer * 100:.4f}%")
    print(f"Overall Word Accuracy (ACC)       : {word_acc * 100:.4f}%")
    print("-" * 60)

    print("Top 20 Cases")
    print(f"{'Truth':<20} | {'Prediction':<20} | {'CER':<6} | {'Match'}")
    print("-" * 60)
    for res in results_log:
        print(f"{res['Truth']:<20} | {res['Pred']:<20} | {res['CER']:<6} | {res['Match']}")
    print("=" * 60)


[1/4] Loading test set data: /root/autodl-fs/Manchu_OCR/test.parquet ...
  > Test set loaded successfully. Total images: 218

[2/4] Loading model from: Um1neko/TrOCR_Manchu ...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

  > Model and Processor assembled and loaded successfully.

[3/4] Starting inference...



Evaluating:   0%|          | 0/218 [00:00<?, ?it/s]


[4/4] Evaluation complete. Calculating final scores...

[TrOCR Test Set Final Results]
Overall Character Error Rate (CER): 29.5110%
Overall Word Error Rate (WER)     : 67.4312%
Overall Word Accuracy (ACC)       : 32.5688%
------------------------------------------------------------
Top 20 Cases
Truth                | Prediction           | CER    | Match
------------------------------------------------------------
alafi                | alafi                | 0.0    | Yes
ujihe                | unggi                | 0.8    | No
acabume              | acabubume            | 0.286  | No
biyai                | biyo                 | 0.4    | No
buhe                 | buhe                 | 0.0    | Yes
afafi                | afa                  | 0.4    | No
dabala               | dalan                | 0.5    | No
afanjiha             | afanjiha             | 0.0    | Yes
dasa                 | dasa                 | 0.0    | Yes
ambasa               | ambakisa             | 0.333  | 